In [1]:
import pystac
import xarray as xr
import zarr

import xpystac

In [2]:
print(zarr.__version__)
print(xr.__version__)
print(xpystac.__version__)
print(pystac.__version__)

3.1.5.dev2734+g596fd7f91
2025.12.0
0.1.dev46+gcba5d10d8.d20251224
1.14.2


In [3]:
xr.backends.list_engines().keys()

dict_keys(['scipy', 'stac', 'store', 'zarr'])

In [4]:
tarStorePathList = [ 'testStore1Dev.tar', 'testStore2Dev.tar' ]

# Single Tar store read with zarr 'TarStore'

In [5]:
with zarr.storage.TarStore( tarStorePathList[0], mode='r' ) as store:
    ds = xr.open_zarr(store).compute()
ds

<xarray.Dataset> Size: 208B
Dimensions:      (time: 2, lat: 3, lon: 3)
Coordinates:
  * time         (time) datetime64[ns] 16B 2023-01-01 2023-01-02
  * lat          (lat) int64 24B 34 35 36
  * lon          (lon) int64 24B -118 -117 -116
Data variables:
    temperature  (time, lat, lon) float64 144B 280.7 294.3 281.7 ... 278.1 279.1
Attributes:
    description:  Sample weather data

# Multiple Tar stores with zarr 'TarStore'

In [6]:
tarStoreList = [ zarr.storage.TarStore( storePath, mode='r')
                     for storePath in tarStorePathList ]

In [7]:
dsFull = xr.open_mfdataset( tarStoreList, engine = 'zarr' )

In [8]:
dsFull

<xarray.Dataset> Size: 352B
Dimensions:        (time: 2, lat: 3, lon: 3)
Coordinates:
  * time           (time) datetime64[ns] 16B 2023-01-01 2023-01-02
  * lat            (lat) int64 24B 34 35 36
  * lon            (lon) int64 24B -118 -117 -116
Data variables:
    temperature    (time, lat, lon) float64 144B dask.array<chunksize=(2, 3, 3), meta=np.ndarray>
    precipitation  (time, lat, lon) float64 144B dask.array<chunksize=(2, 3, 3), meta=np.ndarray>
Attributes:
    description:  Sample weather data

In [9]:
dsFull.compute()

<xarray.Dataset> Size: 352B
Dimensions:        (time: 2, lat: 3, lon: 3)
Coordinates:
  * time           (time) datetime64[ns] 16B 2023-01-01 2023-01-02
  * lat            (lat) int64 24B 34 35 36
  * lon            (lon) int64 24B -118 -117 -116
Data variables:
    temperature    (time, lat, lon) float64 144B 280.7 294.3 ... 278.1 279.1
    precipitation  (time, lat, lon) float64 144B 0.7805 0.3212 ... 0.4721
Attributes:
    description:  Sample weather data

# Accessing assets from STAC catalog and creating 'archiveextension' based assets

In [10]:
ngc4008_collection=pystac.Collection.from_file("https://wwestac.cloud.dkrz.de/stac-fastapi-es/collections/ngc4008")
ngc4008_collection.to_dict()
items=pystac.ItemCollection.from_file("https://wwestac.cloud.dkrz.de/stac-fastapi-es/collections/ngc4008/items")
print(len(items))

10


In [11]:
item = items[1]
item.id
item.to_dict()
item.assets

{'disk': <Asset href=file:///work/bm1235/k203123/nextgems_prefinal/experiments/ngc4008/outdata/ngc4008_P1D_5.zarr>}

In [12]:
item.assets['tape1'] = {'href': tarStorePathList[0],
   'type': 'application/x-tar',
   'archive:format': 'application/x-tar',
   'archive:href': tarStorePathList[0],
   'archive:type': 'application/vnd+zarr'}

In [13]:
item.assets['tape2'] = {'href': tarStorePathList[1],
   'type': 'application/x-tar',
   'archive:format': 'application/x-tar',
   'archive:href': tarStorePathList[1],
   'archive:type': 'application/vnd+zarr'}

In [14]:
item.assets

{'disk': <Asset href=file:///work/bm1235/k203123/nextgems_prefinal/experiments/ngc4008/outdata/ngc4008_P1D_5.zarr>,
 'tape1': {'href': 'testStore1Dev.tar',
  'type': 'application/x-tar',
  'archive:format': 'application/x-tar',
  'archive:href': 'testStore1Dev.tar',
  'archive:type': 'application/vnd+zarr'},
 'tape2': {'href': 'testStore2Dev.tar',
  'type': 'application/x-tar',
  'archive:format': 'application/x-tar',
  'archive:href': 'testStore2Dev.tar',
  'archive:type': 'application/vnd+zarr'}}

# Accessing single STAC asset with 'archiveextension' using xpystac and reading with zarr 'TarStore'

In [15]:
tape_asset=item.assets['tape1']
tape_asset

{'href': 'testStore1Dev.tar',
 'type': 'application/x-tar',
 'archive:format': 'application/x-tar',
 'archive:href': 'testStore1Dev.tar',
 'archive:type': 'application/vnd+zarr'}

In [16]:
ds_tar=xr.open_dataset(pystac.asset.Asset.from_dict(tape_asset),engine="stac")

Asset as input!
Opening tarstore : testStore1Dev.tar


In [17]:
ds_tar

<xarray.Dataset> Size: 208B
Dimensions:      (time: 2, lat: 3, lon: 3)
Coordinates:
  * time         (time) datetime64[ns] 16B 2023-01-01 2023-01-02
  * lat          (lat) int64 24B 34 35 36
  * lon          (lon) int64 24B -118 -117 -116
Data variables:
    temperature  (time, lat, lon) float64 144B ...
Attributes:
    description:  Sample weather data

# Accessing multiple STAC assets with 'archiveextension' using xpystac and reading with zarr 'TarStore'

In [18]:
tape_assets=[ item.assets['tape1'], item.assets['tape2'] ]

In [19]:
tape_assets

[{'href': 'testStore1Dev.tar',
  'type': 'application/x-tar',
  'archive:format': 'application/x-tar',
  'archive:href': 'testStore1Dev.tar',
  'archive:type': 'application/vnd+zarr'},
 {'href': 'testStore2Dev.tar',
  'type': 'application/x-tar',
  'archive:format': 'application/x-tar',
  'archive:href': 'testStore2Dev.tar',
  'archive:type': 'application/vnd+zarr'}]

In [20]:
tape_assetList = [pystac.asset.Asset.from_dict(tape_asset)
                      for tape_asset in tape_assets]

In [21]:
ds_tarList=xr.open_dataset( tape_assetList, engine="stac" )

List of Assets as input!


100%|██████████| 2/2 [00:00<00:00, 20763.88it/s]

Opening tarstore : testStore1Dev.tar
Opening tarstore : testStore2Dev.tar


In [22]:
ds_tarList

<xarray.Dataset> Size: 352B
Dimensions:        (time: 2, lat: 3, lon: 3)
Coordinates:
  * time           (time) datetime64[ns] 16B 2023-01-01 2023-01-02
  * lat            (lat) int64 24B 34 35 36
  * lon            (lon) int64 24B -118 -117 -116
Data variables:
    temperature    (time, lat, lon) float64 144B ...
    precipitation  (time, lat, lon) float64 144B ...
Attributes:
    description:  Sample weather data